# 3.1 — VolumeKernel3D and the Riesz Map

This notebook demonstrates `VolumeKernel3D` — the primary linear functional in
`pygeoinf3d` — and the Riesz duality pairing for `Lebesgue3D`.

A **VolumeKernel3D** implements the inner-product functional
$$\ell_k[f] = \int_\Omega k(\mathbf{x})\, f(\mathbf{x})\, \mathrm{d}V$$
where $k \in L^2(\Omega)$ is the kernel and $f$ is a primal function.

**Sections:**
1. Imports and setup
2. Constructing `VolumeKernel3D`
3. Evaluating the functional on test functions
4. Linearity verification
5. Visualisation mode
6. Multiple kernels (SOLA-style grid)
7. Riesz map and duality pairing
8. Kernel visualisation
9. Summary

---
> **Visualisation mode:** in the plotting sections you can choose `"static"`, `"interactive"`, or `"both"` and re-run the visual cells.

## 1 — Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.testing import assert_allclose

from pygeoinf3d import BoxDomain, Function3D, Lebesgue3D, VolumeKernel3D
from pygeoinf3d.plot import plot_function3d_axis_slice

# Unit cube [0,1]^3
box   = BoxDomain(np.zeros(3), np.ones(3))
space = Lebesgue3D(box)

print("BoxDomain:", box)
print("Volume  :", box.volume)
print("Lebesgue3D.dim (basis-free mode → 0):", space.dim)


## 2 — VolumeKernel3D Construction

`VolumeKernel3D(space, kernel)` constructs the linear functional
$$\ell_k[f] = \int_\Omega k(\mathbf{x})\, f(\mathbf{x})\,\mathrm{d}V$$

**Important:** `kernel` must be a `Function3D` object (not a raw Python callable).
Use `Function3D(box, evaluate_callable=fn)` where `fn` accepts an `(N, 3)` array
and returns an `(N,)` array.


In [ ]:
# Gaussian kernel: k(x) = exp(-||x - c||² / 2σ²)
center = np.array([0.5, 0.5, 0.5])
sigma  = 0.15

kernel_fn = Function3D(
    box,
    evaluate_callable=lambda x: np.exp(
        -np.sum((x - center)**2, axis=1) / (2 * sigma**2)
    )
)

# Construct the VolumeKernel3D functional
vk = VolumeKernel3D(space, kernel_fn)

print("type(vk)    :", type(vk))
print("vk.domain   :", vk.domain)
print("\nPublic methods:")
print([m for m in dir(vk) if not m.startswith("_")])


## 3 — Evaluating the Functional on Test Functions

We evaluate $\ell_k$ on three test functions and numerically verify the result
for the constant function:

$$\ell_k[1] = \int_{[0,1]^3} k(\mathbf{x})\,\mathrm{d}V $$


In [ ]:
# Three test functions on [0,1]^3
f_const = Function3D(box, evaluate_callable=lambda x: np.ones(len(x)))
f_sin   = Function3D(
    box,
    evaluate_callable=lambda x: (
        np.sin(np.pi * x[:, 0]) *
        np.sin(np.pi * x[:, 1]) *
        np.sin(np.pi * x[:, 2])
    )
)
f_poly  = Function3D(box, evaluate_callable=lambda x: x[:, 0] * x[:, 1] * x[:, 2])

# Evaluate the functional
val_const = vk(f_const)
val_sin   = vk(f_sin)
val_poly  = vk(f_poly)

print(f"vk(f_const) = {val_const:.8f}")
print(f"vk(f_sin)   = {val_sin:.8f}")
print(f"vk(f_poly)  = {val_poly:.8f}")

# Numerical verification for f_const:
# vk(f_const) = ∫ k(x) dx  (by direct quadrature)
integral_k = box.integrate_volume(
    lambda x: np.exp(-np.sum((x - center)**2, axis=1) / (2 * sigma**2))
)
print(f"\nDirect integral of k:  {integral_k:.8f}")
print(f"vk(f_const):           {val_const:.8f}")
assert_allclose(val_const, integral_k, rtol=1e-6,
                err_msg="vk(f_const) should equal ∫ k dV")
print("✓  vk(f_const) == ∫ k dV  (passes assert_allclose, rtol=1e-6)")


## 4 — Linearity Verification

$\ell_k$ is a linear functional, so
$$\ell_k[\alpha f + \beta g] = \alpha\,\ell_k[f] + \beta\,\ell_k[g].$$

We verify this numerically with $\alpha = 2,\; \beta = -1$.


In [ ]:
alpha, beta = 2.0, -1.0

# Build α·f_const + β·f_sin as a new Function3D
f_combo = Function3D(
    box,
    evaluate_callable=lambda x: (
        alpha * np.ones(len(x)) +
        beta  * np.sin(np.pi * x[:, 0]) * np.sin(np.pi * x[:, 1]) * np.sin(np.pi * x[:, 2])
    )
)

lhs = vk(f_combo)
rhs = alpha * val_const + beta * val_sin

print(f"vk(α·f_const + β·f_sin) = {lhs:.10f}")
print(f"α·vk(f_const) + β·vk(f_sin) = {rhs:.10f}")
print(f"Difference = {abs(lhs - rhs):.2e}")

assert_allclose(lhs, rhs, atol=1e-10,
                err_msg="Linearity violated")
print("✓  Linearity holds (atol=1e-10)")


In [ ]:
VISUALIZATION_MODE = "interactive"  # change to "interactive" or "both"
INTERACTIVE_AXIS = "z"
INTERACTIVE_N_SLICES = 31
INTERACTIVE_RESOLUTION = 80
INTERACTIVE_FUNCTION_NAMES = ["kernel_sum", "kernel_fn"]

try:
    from pygeoinf3d.plot import plot_function3d_axis_slider
    INTERACTIVE_PLOTTING_AVAILABLE = True
except ImportError:
    plot_function3d_axis_slider = None
    INTERACTIVE_PLOTTING_AVAILABLE = False

valid_modes = {"static", "interactive", "both"}
if VISUALIZATION_MODE not in valid_modes:
    raise ValueError(
        f"VISUALIZATION_MODE must be one of {sorted(valid_modes)}, got {VISUALIZATION_MODE!r}"
    )

print(f"VISUALIZATION_MODE = {VISUALIZATION_MODE!r}")
print(f"Interactive plotting available: {INTERACTIVE_PLOTTING_AVAILABLE}")

## 5 — Visualisation Mode

This notebook supports two plotting styles for the kernel visualisation cells:

- `"static"`: Matplotlib heatmaps embedded directly in the notebook
- `"interactive"`: Plotly slice sliders from `pygeoinf3d.plot`
- `"both"`: render both versions

The interactive option depends on Plotly. If it is not installed, the notebook will print a short message instead of failing.

## 6 — Multiple Kernels: SOLA-style Grid

In SOLA-type inversions we build a bank of localised kernels — one per target
location — and use the resulting measurements to constrain the model.

Here we create a 3×3 grid of Gaussian kernels at different (x, y) centres,
all fixed at $z = 0.5$:
$$k_i(\mathbf{x}) = \exp\!\left(-\frac{\|\mathbf{x} - \mathbf{c}_i\|^2}{2\sigma^2}\right), \quad \mathbf{c}_i \in \{0.25,0.5,0.75\}^2 \times \{0.5\}$$


In [ ]:
sigma_grid = 0.12

centers_xy = [(cx, cy)
              for cy in [0.25, 0.5, 0.75]
              for cx in [0.25, 0.5, 0.75]]
grid_centers = np.array([[cx, cy, 0.5] for cx, cy in centers_xy])

# Build one VolumeKernel3D per grid centre
def make_kernel(c):
    kfn = Function3D(
        box,
        evaluate_callable=lambda x, _c=c: np.exp(
            -np.sum((x - _c)**2, axis=1) / (2 * sigma_grid**2)
        )
    )
    return VolumeKernel3D(space, kfn)

grid_kernels = [make_kernel(c) for c in grid_centers]

# Evaluate all kernels against f_const (which equals 1 everywhere)
vals = np.array([k(f_const) for k in grid_kernels])
print("Kernel values on f_const (3×3 grid):")
print(vals.reshape(3, 3).round(6))

# --- Heatmap ---
fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(vals.reshape(3, 3), origin="lower",
               extent=[0.125, 0.875, 0.125, 0.875],
               vmin=0, cmap="inferno")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("∫ k_i(x) dV  for 3×3 kernel grid\n(applied to f=1, z_c=0.5)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
# Kernel sum: S(x) = sum_i k_i(x)
def kernel_sum_callable(x):
    return sum(
        np.exp(-np.sum((x - c)**2, axis=1) / (2 * sigma_grid**2))
        for c in grid_centers
    )

f_kernel_sum = Function3D(box, evaluate_callable=kernel_sum_callable)
function_map = {
    "kernel_sum": (f_kernel_sum, "Sum of 9 Gaussian kernels"),
    "kernel_fn": (kernel_fn, f"Gaussian kernel  k(x)  (σ = {sigma}, centre = {tuple(center)})"),
}

selected_names = []
for name in INTERACTIVE_FUNCTION_NAMES:
    if name not in function_map:
        raise ValueError(
            f"Unknown function name {name!r}. Choose from {sorted(function_map)}."
        )
    selected_names.append(name)

if VISUALIZATION_MODE in {"static", "both"}:
    fig, ax = plt.subplots(figsize=(5, 4))
    plot_function3d_axis_slice(
        f_kernel_sum,
        axis="z",
        coord=0.5,
        resolution=120,
        show_plot=False,
        title="Sum of 9 Gaussian kernels — slice at z = 0.5",
        ax=ax,
    )
    plt.tight_layout()
    plt.show()
else:
    print(
        "Skipping static kernel-sum slice because VISUALIZATION_MODE='interactive'."
    )

if VISUALIZATION_MODE in {"interactive", "both"}:
    if not INTERACTIVE_PLOTTING_AVAILABLE:
        print(
            "Interactive plotting requested, but Plotly is not installed. "
            "Install pygeoinf3d[interactive] or plotly to enable sliders."
        )
    else:
        print(
            f"Rendering interactive {INTERACTIVE_AXIS}-axis sliders for: "
            f"{', '.join(selected_names)}"
        )
        for name in selected_names:
            fn, label = function_map[name]
            fig_slider = plot_function3d_axis_slider(
                fn,
                axis=INTERACTIVE_AXIS,
                n_slices=INTERACTIVE_N_SLICES,
                resolution=INTERACTIVE_RESOLUTION,
                show_plot=False,
                title=f"{label} — interactive {INTERACTIVE_AXIS}-slice slider",
            )
            fig_slider.show()

## 7 — Riesz Map and Duality Pairing

For a Hilbert space $H$, the **Riesz map** $R : H \to H^*$ sends every primal
element $f \in H$ to its canonical dual:
$$R(f)[g] = \langle f, g \rangle_H = \int_\Omega f(\mathbf{x})\,g(\mathbf{x})\,\mathrm{d}V.$$

In `pygeoinf3d` the map is split into two complementary methods on `Lebesgue3D`:

| Direction | Method | Returns |
|-----------|--------|---------|
| Primal → Dual | `space.to_dual(f)` | `VolumeKernel3D` |
| Dual → Primal | `space.riesz(vk)` | `Function3D` |

Note on naming: `space.riesz(dual)` is the *inverse* Riesz map (dual → primal);
`space.to_dual(f)` is the forward Riesz map (primal → dual).


In [ ]:
print("space.riesz available:", hasattr(space, "riesz"))
print("space.to_dual available:", hasattr(space, "to_dual"))

# --- Forward Riesz: f_sin → VolumeKernel3D ---
dual_sin = space.to_dual(f_sin)
print("\nspace.to_dual(f_sin) →", type(dual_sin))

# The dual element dual_sin evaluates as:
#   dual_sin(g) = ∫ f_sin(x) g(x) dV
# Verify against direct evaluation of vk(f_sin):
val_direct = dual_sin(f_const)
val_cross  = space.inner_product(f_sin, f_const)
print(f"\ndual_sin(f_const)              = {val_direct:.8f}")
print(f"<f_sin, f_const>_L² (direct)  = {val_cross:.8f}")
assert_allclose(val_direct, val_cross, rtol=1e-6,
                err_msg="to_dual duality pairing mismatch")
print("✓  dual_sin(f_const) == <f_sin, f_const>  (rtol=1e-6)")

# --- Inverse Riesz: VolumeKernel3D → Function3D ---
#     space.riesz(vk)  recovers the kernel function
recovered = space.riesz(dual_sin)
print(f"\nspace.riesz(dual_sin) → {type(recovered)}")

# The recovered function should equal f_sin pointwise
xs = box.sample_interior(500)
assert_allclose(recovered(xs), f_sin(xs), rtol=1e-8,
                err_msg="riesz should recover original function")
print("✓  Recovered function equals f_sin pointwise (rtol=1e-8)")

# Pairing between vk (kernel at 0.5) and dual of f_sin:
#   dual_sin(f_kernel) = ∫ f_sin(x) k(x) dV  = vk(f_sin)
pairing_1 = dual_sin(kernel_fn)           # dual_sin evaluates a Function3D
pairing_2 = vk(f_sin)                     # vk evaluates on f_sin
print(f"\ndual_sin(kernel_fn) = {pairing_1:.8f}")
print(f"vk(f_sin)           = {pairing_2:.8f}")
assert_allclose(pairing_1, pairing_2, rtol=1e-6,
                err_msg="Duality symmetry ⟨f_sin, k⟩ == ⟨k, f_sin⟩")
print("✓  ⟨f_sin, k⟩ == ⟨k, f_sin⟩  (symmetry of L² inner product)")


## 8 — Kernel Visualisation

A 2D slice at $z = 0.5$ reveals the Gaussian shape of the kernel function.
The cell below respects `VISUALIZATION_MODE`: it can show the static Matplotlib heatmap, the interactive slider view, or both.

In [ ]:
if VISUALIZATION_MODE in {"static", "both"}:
    fig, ax = plt.subplots(figsize=(5, 4))
    plot_function3d_axis_slice(
        kernel_fn,
        axis="z",
        coord=0.5,
        resolution=150,
        show_plot=False,
        title=f"Gaussian kernel  k(x)  at z = 0.5\n(σ = {sigma}, centre = {tuple(center)})",
        ax=ax,
    )
    plt.tight_layout()
    plt.show()
else:
    print(
        "Skipping static kernel heatmap because VISUALIZATION_MODE='interactive'."
    )

## 9 — Summary

| Feature | Class / Method | Notes |
|---------|---------------|-------|
| Linear functional | `VolumeKernel3D(space, kernel)` | `kernel` must be a `Function3D` |
| Evaluate | `vk(f)` → `float` | inner product $\int k\,f\,\mathrm{d}V$ |
| Forward Riesz | `space.to_dual(f)` → `VolumeKernel3D` | primal → dual |
| Inverse Riesz | `space.riesz(vk)` → `Function3D` | dual → primal |
| Volume integral | `box.integrate_volume(callable)` | callable: `(N,3) → (N,)` |
| Static slice plot | `plot_function3d_axis_slice(fn, axis, coord)` | `fn` must be `Function3D` |
| Interactive slice plot | `plot_function3d_axis_slider(fn, axis, ...)` | Plotly optional dependency |

**Next demo:** [4_providers_3d_demo.ipynb](pygeoinf3D/demos/4_providers_3d_demo.ipynb) — `FunctionProvider3D`, `KernelProvider3D`, and `FunctionToKernelAdapter`.